In [ ]:
import pandas as pd
import sqlite3
import os

In [ ]:
# 1. Paths (Update txt_folder to your actual path)
db_path = r"../data/database/faers_2025.db"
txt_folder = r"../data/raw"

In [ ]:
# 2. Establish a connection to the SQLite database
conn = sqlite3.connect(db_path)

# Define the 7 core tables and the 4 quarters of the year
tables = ["DEMO", "DRUG", "REAC", "THER", "OUTC", "RPSR", "INDI"]
quarters = ["Q1", "Q2", "Q3", "Q4"]
year = "25"

print("🚀 Starting Data Ingestion Pipeline with Out-of-Time Split...")

for table_name in tables:
    print(f"\n--- Processing Table: {table_name} ---")
    
    # We use 'replace' for the first quarter to reset the table, then 'append'
    is_first_insert = True 
    
    for q in quarters:
        file_name = f"{table_name}{year}{q}.txt"
        file_path = os.path.join(txt_folder, file_name)
        
        if os.path.exists(file_path):
            print(f"Loading {file_name}...")
            
            try:
                # Read the file safely
                df = pd.read_csv(file_path, sep="$", low_memory=False, on_bad_lines='skip', encoding='utf-8')
                
                # --- Universal Cleaning ---
                # 1. Standardize column names to lowercase
                df.columns = df.columns.str.lower()
                
                # 2. Drop columns that are entirely empty
                df.dropna(axis=1, how='all', inplace=True)
                
              
                df['is_test_set'] = 1 if q == "Q4" else 0
                
                # 4. Inject into SQLite
                # If it's the very first file for this table, replace old data. Otherwise, append.
                mode = 'replace' if is_first_insert else 'append'
                df.to_sql(table_name.lower(), conn, if_exists=mode, index=False)
                
                print(f" Inserted {len(df)} rows into '{table_name.lower()}' (Test Set: {q == 'Q4'}).")
                is_first_insert = False
                
            except Exception as e:
                print(f" Error processing {file_name}: {e}")
        else:
            print(f" Warning: {file_name} not found at {file_path}")

# Close the database connection
conn.close()